## 01 — Data preparation

This notebook:
- loads one or more teammate JSONL datasets (shared schema)
- validates required columns
- prints a dataset summary
- builds task-specific tables (classification + generation)

Outputs are saved under `najel-data/training/artifacts/`.


In [4]:
from pathlib import Path
import json
import sys

# Make the reusable modules importable
THIS_NOTEBOOK_DIR = Path.cwd()
SRC_DIR = (THIS_NOTEBOOK_DIR.parent / "src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from capstone_training.schema_utils import SchemaSpec
from capstone_training.data_utils import load_jsonl_folder, summarize_incident_df
from capstone_training.dataset_builders import TaskSpec, build_task_dataset


In [5]:
# Point this at a folder containing one or more teammate JSONL files
# Example in this repo: najel-data/data/
DATA_FOLDER = (THIS_NOTEBOOK_DIR.parent.parent / "data").resolve()

schema = SchemaSpec.k8s_incident_v1()
df, summary = load_jsonl_folder(DATA_FOLDER, schema=schema)

print("Loaded files:")
for f in summary.files:
    print("-", f)
print("Rows:", summary.rows)

# Show schema extras (non-fatal)
for fp, rep in summary.schema_reports.items():
    if rep.unexpected_extra:
        print(f"[schema] {fp} unexpected extra columns: {sorted(rep.unexpected_extra)}")

summ = summarize_incident_df(df)
print(json.dumps({k: v for k, v in summ.items() if k != "samples"}, indent=2))
print("\nSample records (first 2):")
print(json.dumps(summ["samples"][:2], indent=2)[:2000])


Loaded files:
- /Users/najelalarcon/Desktop/Multi-Agent-Collaboration-System/najel-data/data/Old_Sanity_Data.jsonl
- /Users/najelalarcon/Desktop/Multi-Agent-Collaboration-System/najel-data/data/k8s_incidents.jsonl
Rows: 1515
{
  "rows": 1515,
  "columns": [
    "scenario_id",
    "namespace",
    "workload_kind",
    "workload_name",
    "container_name",
    "image",
    "created_at",
    "pod_status",
    "waiting_reason",
    "evidence_text",
    "diagnosis_text",
    "fix_plan_text",
    "actions_text",
    "verification_text",
    "rollback_text",
    "id",
    "event_reason",
    "missing_secret",
    "failure_phase",
    "noise_level",
    "event_message",
    "error_message",
    "symptom_family",
    "cluster_id",
    "restart_count",
    "missing_key",
    "root_cause_family",
    "oom_killed",
    "event_type",
    "difficulty",
    "__source_file"
  ],
  "scenario_distribution": {
    "failedscheduling_insufficient_memory": 505,
    "failedscheduling_insufficient_cpu": 505,

In [6]:
# Build task datasets
clf_spec = TaskSpec(
    name="scenario_classification",
    task_type="classification",
    input_col="evidence_text",
    target_col="scenario_id",
)

gen_spec = TaskSpec(
    name="remediation_generation",
    task_type="generation",
    input_col="evidence_text",
    target_col="scenario_id",
    use_chat_format=True,
)

df_clf = build_task_dataset(df, clf_spec)
df_gen = build_task_dataset(df, gen_spec)

out_dir = (THIS_NOTEBOOK_DIR.parent / "artifacts" / "datasets").resolve()
out_dir.mkdir(parents=True, exist_ok=True)

clf_path = out_dir / "classification.parquet"
gen_path = out_dir / "generation.parquet"

df_clf.to_parquet(clf_path, index=False)
df_gen.to_parquet(gen_path, index=False)

print("Saved:")
print("-", clf_path)
print("-", gen_path)
print("classification columns:", [c for c in ["prompt", "label", "scenario_id", "__source_file"] if c in df_clf.columns])
print("generation columns:", [c for c in ["prompt", "target", "text", "scenario_id", "__source_file"] if c in df_gen.columns])


Saved:
- /Users/najelalarcon/Desktop/Multi-Agent-Collaboration-System/najel-data/training/artifacts/datasets/classification.parquet
- /Users/najelalarcon/Desktop/Multi-Agent-Collaboration-System/najel-data/training/artifacts/datasets/generation.parquet
classification columns: ['prompt', 'label', 'scenario_id', '__source_file']
generation columns: ['prompt', 'target', 'text', 'scenario_id', '__source_file']
